# PHẦN IV — KHỞI TẠO CONTROLLER VM (RUNBOOK RÚT GỌN)

Notebook rút gọn: chỉ lệnh + comment ngắn.

## BƯỚC 17 — Cloud-init Controller 1

In [ ]:
# Controller 1: enp1s0=10.0.0.11, enp2s0=192.168.150.21
cat > /tmp/ctrl1-user-data << EOF
#cloud-config
hostname: controller-1
fqdn: controller-1.openstack.lab
manage_etc_hosts: true
password: 123.abc
chpasswd: {expire: false}
ssh_pwauth: true
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses: ["10.0.0.11/24"]
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.21"
    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses: ["192.168.150.21/24"]
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]
runcmd:
  - ip link set enp1s0 mtu 1450 || true
EOF

cat > /tmp/ctrl1-meta-data << EOF
instance-id: controller-1
local-hostname: controller-1
EOF

cloud-localds /tmp/ctrl1-seed.iso /tmp/ctrl1-user-data /tmp/ctrl1-meta-data
ls -lh /tmp/ctrl1-seed.iso

## BƯỚC 17 — Cloud-init Controller 2

In [ ]:
# Controller 2: enp1s0=10.0.0.12, enp2s0=192.168.150.22
cat > /tmp/ctrl2-user-data << EOF
#cloud-config
hostname: controller-2
fqdn: controller-2.openstack.lab
manage_etc_hosts: true
password: 123.abc
chpasswd: {expire: false}
ssh_pwauth: true
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses: ["10.0.0.12/24"]
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.22"
    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses: ["192.168.150.22/24"]
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]
runcmd:
  - ip link set enp1s0 mtu 1450 || true
EOF

cat > /tmp/ctrl2-meta-data << EOF
instance-id: controller-2
local-hostname: controller-2
EOF

cloud-localds /tmp/ctrl2-seed.iso /tmp/ctrl2-user-data /tmp/ctrl2-meta-data
ls -lh /tmp/ctrl2-seed.iso

## BƯỚC 17 — Cloud-init Controller 3

In [ ]:
# Controller 3: enp1s0=10.0.0.13, enp2s0=192.168.150.23
cat > /tmp/ctrl3-user-data << EOF
#cloud-config
hostname: controller-3
fqdn: controller-3.openstack.lab
manage_etc_hosts: true
password: 123.abc
chpasswd: {expire: false}
ssh_pwauth: true
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses: ["10.0.0.13/24"]
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.23"
    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses: ["192.168.150.23/24"]
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]
runcmd:
  - ip link set enp1s0 mtu 1450 || true
EOF

cat > /tmp/ctrl3-meta-data << EOF
instance-id: controller-3
local-hostname: controller-3
EOF

cloud-localds /tmp/ctrl3-seed.iso /tmp/ctrl3-user-data /tmp/ctrl3-meta-data
ls -lh /tmp/ctrl3-seed.iso

## BƯỚC 18 — Tạo Controller VM1

In [ ]:
# Chạy trên Server A
cd /var/lib/libvirt/templates
sudo lvcreate -V 100G --thin -n ctrl-vm-1 vg-lab/thin-pool
sudo dd if=/var/lib/libvirt/templates/tpl-controller.img of=/dev/vg-lab/ctrl-vm-1 bs=4M status=progress

virt-install \
  --name controller-vm-1 \
  --memory 10240 \
  --vcpus 2 \
  --disk /dev/vg-lab/ctrl-vm-1,format=raw,bus=virtio \
  --disk /tmp/ctrl1-seed.iso,device=cdrom \
  --os-variant ubuntu24.04 \
  --network bridge=br-mgmt,model=virtio \
  --network bridge=br-lab,model=virtio \
  --graphics vnc,listen=0.0.0.0 \
  --noautoconsole \
  --import

virsh list --all

## BƯỚC 18 — Tạo Controller VM2

In [ ]:
# Chạy trên Server B
cd /var/lib/libvirt/templates
sudo lvcreate -V 100G --thin -n ctrl-vm-2 vg-lab/thin-pool
sudo dd if=/var/lib/libvirt/templates/tpl-controller.img of=/dev/vg-lab/ctrl-vm-2 bs=4M status=progress

virt-install \
  --name controller-vm-2 \
  --memory 10240 \
  --vcpus 2 \
  --disk /dev/vg-lab/ctrl-vm-2,format=raw,bus=virtio \
  --disk /tmp/ctrl2-seed.iso,device=cdrom \
  --os-variant ubuntu24.04 \
  --network bridge=br-mgmt,model=virtio \
  --network bridge=br-lab,model=virtio \
  --graphics vnc,listen=0.0.0.0 \
  --noautoconsole \
  --import

virsh list --all

## BƯỚC 18 — Tạo Controller VM3

In [ ]:
# Chạy trên Server C
cd /var/lib/libvirt/templates
sudo lvcreate -V 100G --thin -n ctrl-vm-3 vg-lab/thin-pool
sudo dd if=/var/lib/libvirt/templates/tpl-controller.img of=/dev/vg-lab/ctrl-vm-3 bs=4M status=progress

virt-install \
  --name controller-vm-3 \
  --memory 10240 \
  --vcpus 2 \
  --disk /dev/vg-lab/ctrl-vm-3,format=raw,bus=virtio \
  --disk /tmp/ctrl3-seed.iso,device=cdrom \
  --os-variant ubuntu24.04 \
  --network bridge=br-mgmt,model=virtio \
  --network bridge=br-lab,model=virtio \
  --graphics vnc,listen=0.0.0.0 \
  --noautoconsole \
  --import

virsh list --all

## BƯỚC 18 — Verify Controller VM

In [ ]:
virsh list --all
virsh domifaddr controller-vm-1 || true
virsh domifaddr controller-vm-2 || true
virsh domifaddr controller-vm-3 || true

ssh ubuntu@192.168.150.21 hostname
ssh ubuntu@192.168.150.22 hostname
ssh ubuntu@192.168.150.23 hostname

ssh ubuntu@10.0.0.11 hostname
ssh ubuntu@10.0.0.12 hostname
ssh ubuntu@10.0.0.13 hostname

## BƯỚC 18 — Verify trong VM

In [ ]:
hostname
ip addr show enp1s0
ip addr show enp2s0
ip route
docker --version
ls -lh /opt/kolla-venv/bin/kolla-ansible
ping -c 3 192.168.150.254
ping -c 3 8.8.8.8